<a href="https://colab.research.google.com/github/BriiceBr/Projeto-FTTx/blob/main/Simulador_FttX_(Steiner).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#include<stdio.h>
#include<stdlib.h>
#include<time.h>
#include<math.h>
#include<string.h>

#define TAM_MAP 52     // Total de pontos do mapa (OLT + postes + clientes)
#define TAM_P 100       // Tamanho da populacao
#define TAX_MUTACAO 10  // Taxa de mutacao

struct poste {
    int id;
    float x;
    float y;
    int tipo; // 0 = OLT, 1 = Poste de Passagem, 2 = Cliente
};

struct individuo {
    int pai[TAM_MAPA];       // O vetor que cria a arvore de conexoes
    double distancia_total;  // O gasto total de cabo em metros
    double atenuacao_maxima;
};

struct poste mapa[TAM_MAPA];
int matriz_arcos[TAM_MAPA][TAM_MAPA]; // O mapa real das ruas: 1 = tem cabo, 0 = nao tem rua

struct individuo pop[TAM_P];
struct individuo pop_nova[TAM_P];

// Funcao para ler os arq dos arcos e nós
void ler_instancia_fttx(char nome_arq[]) {
    FILE* arq = fopen(nome_arq, "r");
    if (arq == NULL) {
        printf("Erro ao abrir arquivo %s!\n", nome_arq);
        exit(1);
    }
    char linhas[256];
    int lendo_nos = 0;
    int lendo_arcos = 0;

    while (fgets(linhas, sizeof(linhas), arq)) {
        // Identifica quando comeca a ler os nós
        if (strstr(linhas, "Nodes") != NULL) {
            lendo_nos = 1;
            lendo_arcos = 0;
            continue;
        }
        // Desliga a leitura de nos e passa para ler os arcos
        if (strstr(linhas, "Arcs") != NULL) {
            lendo_nos = 0;
            lendo_arcos = 1;
            continue;
        }
        // Desliga a leitura de arcos caso encontre outras secoes
        if (strstr(linhas, "Splitters") != NULL || strstr(linhas, "BalancedSplitters") != NULL) {
            lendo_arcos = 0;
        }

        // LEITURA DOS NÓS
        if (lendo_nos == 1) {
            int id_lido;
            float x, y;

            // Le o ID e depois as coordenadas X e Y
            if (sscanf(linhas, "%d %f %f", &id_lido, &x, &y) == 3) {
                // ID do txt diferente
                int id_correto = id_lido - 1;
                mapa[id_correto].id = id_correto;
                mapa[id_correto].x = x;
                mapa[id_correto].y = y;

                // Classifica o tipo de ponto
                if (id_lido == 1) {
                    mapa[id_correto].tipo = 0; // OLT
                }
                // Adaptando os ultimos 12 pontos como clientes
                else if (id_lido >= (TAM_MAPA - 11) && id_lido <= TAM_MAPA) {
                    mapa[id_correto].tipo = 2; // Cliente Final
                }
                else {
                    mapa[id_correto].tipo = 1; // Poste de Passagem
                }
            }
        }

        // LEITURA DOS ARCOS
        if (lendo_arcos == 1) {
            int origem, destino;
            if (sscanf(linhas, "%d %d", &origem, &destino) == 2) {
                // Preenche a matriz marcando 1 onde existe rua
                matriz_arcos[origem - 1][destino - 1] = 1;
            }
        }
    }

    fclose(arq);
}

// Essa funcao usa o algoritmo de prim randomizado, conforme “ALGORITMOS EVOLUTIVOS APLICADOS AO PROBLEMA DA ÁRVORE DE STEINER EUCLIDIANO"
void gerar_primeira_pop(struct individuo* ind) {
    int i, j;
    int visitado[TAM_MAPA];

    // Limpa o vetor de pais e o vetor de controle
    for(i = 0; i < TAM_MAPA; i++) {
        ind->pai[i] = -1;
        visitado[i] = 0;
    }

    // A raiz da arvore e a OLT (indice 0)
    visitado[0] = 1;
    ind->pai[0] = -1; // OLT nao tem pai

    int nos_conectados = 1;

    // Laco para conectar todos os outros nos do mapa
    while (nos_conectados < TAM_MAPA) {
        // Vetores temporarios para guardar as opcoes de ruas
        int cand_origem[TAM_MAPA * TAM_MAPA];
        int cand_destino[TAM_MAPA * TAM_MAPA];
        int num_candidatos = 0;

        // Procura todas as ruas validas saindo de quem ja esta na rede para quem ainda nao esta
        for (i = 0; i < TAM_MAPA; i++) {
            if (visitado[i] == 1) {
                for (j = 0; j < TAM_MAPA; j++) {
                    if (matriz_arcos[i][j] == 1 && visitado[j] == 0) {
                        cand_origem[num_candidatos] = i;
                        cand_destino[num_candidatos] = j;
                        num_candidatos++;
                    }
                }
            }
        }
        if (num_candidatos == 0) {
            break;
        }
        // Sorteio de rua
        int sorteio = rand() % num_candidatos;
        int origem_escolhida = cand_origem[sorteio];
        int destino_escolhido = cand_destino[sorteio];

        // Conecta o novo poste na rede anotando de onde o cabo veio
        ind->pai[destino_escolhido] = origem_escolhida;
        visitado[destino_escolhido] = 1;
        nos_conectados++;
    }
}

// Calculo de distancia com pitagoras igual do outro simulador
double pit(struct poste p1, struct poste p2) {
    float dx = p1.x - p2.x;
    float dy = p1.y - p2.y;
    return sqrt((dx * dx) + (dy * dy));
}

// A nova funcao de avaliacao
void avaliar_individuo(struct individuo *ind) {
    int i;
    int filhos[TAM_MAPA];
    int removeu_alguem;

    // Repete o corte de cabos ate que nenhum poste inutil sobre
    do {
        removeu_alguem = 0;
        // Zera a contagem de filhos
        for(i = 0; i < TAM_MAPA; i++) {
            filhos[i] = 0;
        }
        // Conta quantos filhos cada no possui na arvore atual
        for(i = 0; i < TAM_MAPA; i++) {
            if (ind->pai[i] != -1) {
                filhos[ind->pai[i]]++;
            }
        }
        // Procura postes de passagem (tipo 1) que nao tem filhos
        for(i = 0; i < TAM_MAPA; i++) {
            if (mapa[i].tipo == 1 && ind->pai[i] != -1 && filhos[i] == 0) {
                ind->pai[i] = -1; // corta
                removeu_alguem = 1;
            }
        }
    } while(removeu_alguem == 1);

    // Distancia total do cabeameto
    double soma_metros = 0.0;
    for(i = 0; i < TAM_MAPA; i++) {
        if (ind->pai[i] != -1) {
            soma_metros += pit(mapa[i], mapa[ind->pai[i]]);
        }
    }
    ind->distancia_total = soma_metros;

    // Atenuacao
    ind->atenuacao_maxima = 0.0;
    // O calculo das perdas em decibeis
}

/// Torneio
struct individuo torneio(struct individuo pop_atual[]) {
    int s1 = rand() % TAM_P;
    int s2 = rand() % TAM_P;
    int s3 = rand() % TAM_P;
    struct individuo vencedor = pop_atual[s1];
    if (pop_atual[s2].distancia_total < vencedor.distancia_total) {
        vencedor = pop_atual[s2];
    }
    if (pop_atual[s3].distancia_total < vencedor.distancia_total) {
        vencedor = pop_atual[s3];
    }
    return vencedor;
}

/// Roleta
struct individuo roleta(struct individuo pop_atual[]) {
    int i;
    double soma_avaliacoes = 0.0;
    double fatias[TAM_P];
    // Cria as fatias
    for (i = 0; i < TAM_P; i++) {
        if (pop_atual[i].distancia_total == 0) {
            fatias[i] = 0.0;
        } else {
            fatias[i] = 1.0 / pop_atual[i].distancia_total;
        }
        soma_avaliacoes += fatias[i];
    }
    // Gira a roleta
    double giro = ((double)rand() / RAND_MAX) * soma_avaliacoes;
    double acumulador = 0.0;
    // Verifica onde a bolinha da roleta parou
    for (i = 0; i < TAM_P; i++) {
        acumulador += fatias[i];
        if (acumulador >= giro) {
            return pop_atual[i];
        }
    }
    return pop_atual[TAM_P - 1];
}

/// Mutacao
void mutacao_arvore(struct individuo *ind) {
    // Sorteia a chance de a mutacao ocorrer (0 a 99)
    int chance = rand() % 100;

    if (chance < TAX_MUTACAO) {
        // Sorteia um no aleatorio para mutar (ignora a OLT no indice 0)
        int no_mutado = (rand() % (TAM_MAPA - 1)) + 1;

        int vizinhos_validos[TAM_MAPA];
        int num_vizinhos = 0;
        int i;

        // Varre o mapa procurando outras opcoes de ruas para este poste
        for (i = 0; i < TAM_MAPA; i++) {
            // Regras: 1. Existe rua fisica? 2. O vizinho ja faz parte da rede? 3. Nao e o pai atual?
            if (matriz_arcos[no_mutado][i] == 1 && ind->pai[i] != -1 && i != ind->pai[no_mutado]) {
                vizinhos_validos[num_vizinhos] = i;
                num_vizinhos++;
            }
        }

        // Se encontrou uma nova rota viavel, troca o pai e altera a topologia da rede
        if (num_vizinhos > 0) {
            int sorteio = rand() % num_vizinhos;
            ind->pai[no_mutado] = vizinhos_validos[sorteio];
        }
    }
}
main () {







}

SyntaxError: invalid character '“' (U+201C) (1003698664.py, line 99)